# Lecture: Anomaly Detection with Autoencoders

An autoencoder trained exclusively on **normal** data learns to reconstruct that data well. When presented with an **anomalous** input it has never seen during training, the reconstruction will be poor — the model lacks the latent representation to reproduce it accurately.

This makes the **reconstruction error** a natural anomaly score:

$$\text{score}(\mathbf{x}) = \|\mathbf{x} - \hat{\mathbf{x}}\|^2$$

- Low score → input looks like training data → **normal**
- High score → input is unfamiliar → **anomaly**

No anomaly labels are required during training — this is a form of **self-supervised learning**. The normal class itself provides all the supervision.

**Setup:** We train the autoencoder only on digit **0** from MNIST. At test time we feed it all 10 digit classes and measure reconstruction error. Digit 0 should score low; all other digits should score high.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/Autoencoder.py ./

## Data Preparation

We filter the MNIST training set to keep only digit **0**. The test set retains all 10 classes so we can evaluate the anomaly score across the full digit distribution.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

NORMAL_CLASS = 0

transform = transforms.Compose([transforms.ToTensor()])

train_full = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_full  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Keep only the normal class for training
normal_idx  = [i for i, (_, y) in enumerate(train_full) if y == NORMAL_CLASS]
train_normal = Subset(train_full, normal_idx)

train_loader = DataLoader(train_normal, batch_size=256, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_full,   batch_size=256, shuffle=False, num_workers=0)

print(f"Training samples (digit {NORMAL_CLASS} only): {len(train_normal)}")
print(f"Test samples (all digits):                    {len(test_full)}")

## Model

We reuse the convolutional autoencoder from notebook 63. The latent dimension is set to **16** — large enough to represent digit 0 well, but small enough to prevent the model from memorising unseen shapes.

In [ ]:
from Autoencoder import Autoencoder

LATENT_DIM = 16
model = Autoencoder(latent_dim=LATENT_DIM).to(device)

_x = torch.zeros(4, 1, 28, 28).to(device)
print("Reconstruction shape:", model(_x).shape)

## Training

Training is identical to the standard autoencoder — MSE reconstruction loss, only on digit 0. The model never sees any other digit class.

In [ ]:
import torch.optim as optim
import torch.nn.functional as F

optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, _ in train_loader:
        x = x.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = F.mse_loss(model(x), x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:2d}, Loss: {total_loss / len(train_loader):.6f}")

In [ ]:
model.save_model(path="models/anomaly_ae_mnist.pth")

If you do not want to train, load the pre-trained model (latent_dim=16, digit 0 only, 20 epochs).

In [ ]:
import torch
from Autoencoder import Autoencoder

device = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_DIM = 16

model = Autoencoder(latent_dim=LATENT_DIM).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/anomaly_ae_mnist.pth", device=device)

## Reconstruction Quality: Normal vs. Anomalous

We first compare reconstructions qualitatively. The model should reconstruct digit 0 faithfully and produce distorted or digit-0-like outputs for all other classes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()

# Pick one example per digit class from the test set
examples = {}
for x, y in test_full:
    label = y if isinstance(y, int) else y.item()
    if label not in examples:
        examples[label] = x
    if len(examples) == 10:
        break

digits   = sorted(examples.keys())
x_batch  = torch.stack([examples[d] for d in digits]).to(device)

with torch.no_grad():
    x_hat = model(x_batch)
    scores = F.mse_loss(x_hat, x_batch, reduction="none").mean(dim=(1, 2, 3)).cpu().numpy()

fig, axes = plt.subplots(3, 10, figsize=(15, 5))

for i, d in enumerate(digits):
    axes[0, i].imshow(x_batch[i].squeeze().cpu(), cmap="gray")
    axes[1, i].imshow(x_hat[i].squeeze().cpu(),   cmap="gray")
    axes[2, i].axis("off")
    axes[2, i].text(0.5, 0.5, f"{scores[i]:.4f}",
                    ha="center", va="center", fontsize=9,
                    color="green" if d == NORMAL_CLASS else "red",
                    transform=axes[2, i].transAxes)
    for row in range(2):
        axes[row, i].axis("off")
        axes[row, i].set_title(f"Digit {d}", fontsize=8)

axes[0, 0].set_ylabel("Input",          fontsize=9)
axes[1, 0].set_ylabel("Reconstruction", fontsize=9)
axes[2, 0].set_ylabel("MSE score",      fontsize=9)

plt.suptitle(f"Normal class: digit {NORMAL_CLASS}  |  green = normal, red = anomaly", fontsize=10)
plt.tight_layout()
plt.show()

## Anomaly Score Distribution

We compute the reconstruction error for every test sample and plot the score distribution per digit class. A good anomaly detector shows:
- **Digit 0**: scores concentrated at low values
- **All other digits**: scores shifted to higher values with minimal overlap

In [ ]:
model.eval()

all_scores = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        x_hat = model(x)
        score = F.mse_loss(x_hat, x, reduction="none").mean(dim=(1, 2, 3))
        all_scores.append(score.cpu())
        all_labels.append(y)

all_scores = torch.cat(all_scores).numpy()
all_labels = torch.cat(all_labels).numpy()

fig, ax = plt.subplots(figsize=(10, 4))

for d in range(10):
    mask = all_labels == d
    label = f"{d} (normal)" if d == NORMAL_CLASS else str(d)
    lw    = 2.5 if d == NORMAL_CLASS else 1.0
    ax.hist(all_scores[mask], bins=60, alpha=0.5, label=label,
            density=True, linewidth=lw,
            color="green" if d == NORMAL_CLASS else None)

ax.set_xlabel("Reconstruction error (MSE)")
ax.set_ylabel("Density")
ax.set_title("Anomaly score distribution per digit class")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## ROC Curve and AUROC

To quantify detection performance we treat this as a binary classification problem: digit 0 = normal (label 0), all other digits = anomaly (label 1). The **AUROC** (Area Under the ROC Curve) measures how well the reconstruction error separates the two groups — 0.5 is random chance, 1.0 is perfect separation.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

binary_labels = (all_labels != NORMAL_CLASS).astype(int)  # 0 = normal, 1 = anomaly

auroc = roc_auc_score(binary_labels, all_scores)
fpr, tpr, _ = roc_curve(binary_labels, all_scores)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"AUROC = {auroc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Random")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title(f"ROC Curve — digit {NORMAL_CLASS} vs. all others")
ax.legend()
plt.tight_layout()
plt.show()

print(f"AUROC: {auroc:.4f}")

## Threshold-Based Detection

In practice a fixed threshold $\tau$ is chosen to classify each sample as normal or anomalous:

$$\hat{y} = \begin{cases} \text{normal} & \text{if } \text{score}(\mathbf{x}) \leq \tau \\ \text{anomaly} & \text{if } \text{score}(\mathbf{x}) > \tau \end{cases}$$

We choose $\tau$ as the 95th percentile of the normal class scores — i.e. we accept 5% false positives on normal data by design.

In [ ]:
normal_scores = all_scores[all_labels == NORMAL_CLASS]
threshold = np.percentile(normal_scores, 95)
print(f"Threshold (95th percentile of normal scores): {threshold:.6f}")

predicted_anomaly = all_scores > threshold
true_anomaly      = all_labels != NORMAL_CLASS

tp = (predicted_anomaly &  true_anomaly).sum()
fp = (predicted_anomaly & ~true_anomaly).sum()
tn = (~predicted_anomaly & ~true_anomaly).sum()
fn = (~predicted_anomaly &  true_anomaly).sum()

print(f"True positives  (anomaly correctly flagged): {tp}")
print(f"False positives (normal incorrectly flagged): {fp}")
print(f"True negatives  (normal correctly passed):   {tn}")
print(f"False negatives (anomaly missed):             {fn}")
print(f"Precision: {tp / (tp + fp):.3f}")
print(f"Recall:    {tp / (tp + fn):.3f}")